# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

# Desarrollo — NFCorpus
**Estudiante:** Christian Ayala  
**Corpus asignado:** NFCorpus (BEIR), en inglés.

## 1. Carga de datos
Usamos los documentos completos e IDs originales de NFCorpus. Los archivos están en
`Datos/raw/nfcorpus/`: `corpus.jsonl`, `queries.jsonl` y `qrels/{train,dev,test}.tsv`.
Se evalúan las 323 consultas de test con sus qrels; no se usa test para ajustar parámetros.
Fuentes: [NFCorpus](https://huggingface.co/datasets/BeIR/nfcorpus) y
[qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels). Instalar dependencias si faltan:
`%pip install -r requirements.txt`.

Para reproducir desde cero, colocar el [ZIP oficial de BEIR](https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip)
en `Datos/raw/nfcorpus.zip` y ejecutar todas las celdas desde la carpeta del notebook.
Los archivos faltantes se extraen automáticamente.


In [63]:
from pathlib import Path
import zipfile
import pandas as pd
from IPython.display import display

BASE_DIR = Path.cwd().resolve()
DATA_DIR = BASE_DIR / "Datos" / "raw" / "nfcorpus"
files = ["corpus.jsonl", "queries.jsonl", "qrels/train.tsv", "qrels/dev.tsv", "qrels/test.tsv"]
for name in files:
    target = DATA_DIR / name
    if not target.exists():
        with zipfile.ZipFile(BASE_DIR / "Datos/raw/nfcorpus.zip") as archive:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(f"nfcorpus/{name}"))

corpus = pd.read_json(DATA_DIR / "corpus.jsonl", lines=True, dtype={"_id": str})
queries = pd.read_json(DATA_DIR / "queries.jsonl", lines=True, dtype={"_id": str})
qrels = {split: pd.read_csv(DATA_DIR / "qrels" / f"{split}.tsv", sep="\t",
                            dtype={"query-id": str, "corpus-id": str})
         for split in ["train", "dev", "test"]}
document_ids = set(corpus["_id"])
assert corpus["_id"].is_unique and queries["_id"].is_unique
for table in qrels.values():
    assert set(table["corpus-id"]).issubset(document_ids)
    assert set(table["query-id"]).issubset(set(queries["_id"]))
    assert not table.duplicated(["query-id", "corpus-id"]).any()
print(f"Documentos: {len(corpus):,} | Consultas: {len(queries):,}")
display(pd.DataFrame([{"Partición": split, "Consultas": table["query-id"].nunique(),
                       "Juicios": len(table)} for split, table in qrels.items()]))


Documentos: 3,633 | Consultas: 3,237


,Partición,Consultas,Juicios
0,train,2590,110575
1,dev,324,11385
2,test,323,12334


## 2. Preprocesamiento e índice invertido
Unimos título y texto, pasamos a minúsculas, tokenizamos, retiramos stopwords inglesas
y aplicamos Porter stemming. Conservamos números, decimales y las negaciones `no`,
`nor`, `not`. Ambos modelos usan esta misma representación, sin dividir documentos.
El índice almacena `término → {ID original: frecuencia}`; DF cuenta documentos y
la longitud media se usa en BM25. Stemming no equivale a comprender sinónimos.


In [64]:
import re
from collections import Counter, defaultdict
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:\.[0-9]+)?")
NEGATIONS = {"no", "nor", "not"}
STOP_WORDS = frozenset(ENGLISH_STOP_WORDS) - NEGATIONS
stemmer = PorterStemmer(mode=PorterStemmer.NLTK_EXTENSIONS)


def tokenizar(text):
    return TOKEN_PATTERN.findall(str(text).lower())


def preprocesar(text):
    return [stemmer.stem(token) for token in tokenizar(text)
            if token not in STOP_WORDS]



documents = corpus.copy()
title = documents["title"].fillna("").astype(str).str.strip()
text = documents["text"].fillna("").astype(str).str.strip()
documents["content"] = (title + " " + text).str.replace(r"\s+", " ", regex=True).str.strip()
documents["tokens"] = documents["content"].map(preprocesar)
documents["token_count"] = documents["tokens"].str.len()


inverted_index = defaultdict(dict)
document_counts = {}
document_lengths = {}
for document_id, tokens in zip(documents["_id"], documents["tokens"], strict=True):
    counts = Counter(tokens)
    document_counts[document_id] = counts
    document_lengths[document_id] = len(tokens)
    for term, frequency in counts.items():
        inverted_index[term][document_id] = frequency
inverted_index = dict(inverted_index)
vocabulary = sorted(inverted_index)
term_to_column = {term: column for column, term in enumerate(vocabulary)}
document_frequency = {term: len(postings) for term, postings in inverted_index.items()}
N = len(documents)
avgdl = sum(document_lengths.values()) / N


print(f"Vocabulario: {len(vocabulary):,} términos | Longitud media: {avgdl:.2f} tokens")
display(pd.DataFrame({"Original": documents["content"].head(2).str.slice(0, 150),
                      "Tokens procesados (muestra)": documents["tokens"].head(2).map(lambda t: t[:20])}))


Vocabulario: 20,984 términos | Longitud media: 156.86 tokens


,Original,Tokens procesados (muestra)
0,Statin Use and Breast Cancer Survival: A Natio...,"[statin, use, breast, cancer, surviv, nationwi..."
1,Statin use after diagnosis of breast cancer an...,"[statin, use, diagnosi, breast, cancer, surviv..."


## 3. TF-IDF y coseno
Usamos TF cruda e IDF suavizado $idf(t)=1+\ln((N+1)/(df(t)+1))$, variante vista en clase.
Los pesos son $tf\cdot idf$. Normalizamos documentos y consulta por su norma L2;
su producto punto es el coseno. Vocabulario e IDF se calculan solo con el corpus.
Se usa una matriz dispersa para almacenar los pesos no nulos.

Los rankings devuelven hasta `top_k` scores positivos, ordenados de mayor a menor;
los empates se resuelven por ID. Sin términos conocidos, la lista es vacía.
El snippet es una vista previa: el cálculo usa el documento completo.


In [65]:
import numpy as np
from scipy.sparse import csr_matrix, diags
ordered_document_ids = documents["_id"].tolist()
document_to_row = {doc_id: row for row, doc_id in enumerate(ordered_document_ids)}
idf_vector = np.array([1 + np.log((N + 1) / (document_frequency[term] + 1))
                       for term in vocabulary], dtype=np.float64)
rows, columns, values = [], [], []
for term, postings in inverted_index.items():
    column = term_to_column[term]
    for doc_id, frequency in postings.items():
        rows.append(document_to_row[doc_id])
        columns.append(column)
        values.append(frequency * idf_vector[column])
weighted_matrix = csr_matrix((values, (rows, columns)), shape=(N, len(vocabulary)), dtype=np.float64)
document_norms = np.sqrt(weighted_matrix.multiply(weighted_matrix).sum(axis=1)).A1
inverse_norms = np.divide(1., document_norms, out=np.zeros_like(document_norms), where=document_norms > 0)
tfidf_matrix = (diags(inverse_norms) @ weighted_matrix).tocsr()


def vectorizar_consulta(query):
    if not isinstance(query, str):
        raise TypeError("query debe ser texto")
    counts = Counter(preprocesar(query))
    cols, vals = [], []
    for term, frequency in counts.items():
        if term in term_to_column:
            column = term_to_column[term]
            cols.append(column)
            vals.append(frequency * idf_vector[column])
    vector = csr_matrix((vals, ([0] * len(cols), cols)), shape=(1, len(vocabulary)), dtype=np.float64)
    norm = np.sqrt(vector.multiply(vector).sum())
    return vector / norm if norm > 0 else vector


def scores_tfidf(query):
    vector = vectorizar_consulta(query)
    return (tfidf_matrix @ vector.T).toarray().ravel()


def buscar_tfidf(query, top_k=10):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k debe ser un entero positivo")
    scores = scores_tfidf(query)
    valid = np.flatnonzero(scores > 0)
    ids = np.asarray(ordered_document_ids)
    order = np.lexsort((ids[valid], -scores[valid]))
    selected = valid[order[:top_k]]
    results = []
    for rank, row in enumerate(selected, 1):
        document = documents.iloc[int(row)]
        results.append({"rank": rank, "document_id": str(document["_id"]),
                        "score": float(scores[row]), "title": str(document["title"]),
                        "snippet": str(document["content"])[:240]})
    return {"model": "tfidf", "results": results}


## 4. BM25

BM25 suma los aportes de los términos distintos de la consulta:

$$BM25(q,d)=\sum_{t\in q}idf(t)\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/avgdl)}$$

Usamos la variante positiva de IDF de clase:

$$idf(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)$$

`k1=1.2` controla la saturación de TF y `b=0.75` la normalización por longitud.
Son los parámetros del ejercicio de clase; no se optimizan con test.
Usamos los mismos documentos, tokens e índice que TF-IDF.
Los scores entre modelos no tienen la misma escala.


In [66]:
K1 = 1.2
B = 0.75
bm25_idf = {term: np.log(1 + (N - df + 0.5) / (df + 0.5))
            for term, df in document_frequency.items()}


def buscar_bm25(query, top_k=10):
    if not isinstance(query, str):
        raise TypeError("query debe ser texto")
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k debe ser un entero positivo")
    scores = defaultdict(float)
    for term in set(preprocesar(query)):
        for doc_id, frequency in inverted_index.get(term, {}).items():
            norm = 1 - B + B * document_lengths[doc_id] / avgdl
            scores[doc_id] += bm25_idf[term] * frequency * (K1 + 1) / (frequency + K1 * norm)
    ranking = sorted(scores.items(), key=lambda item: (-item[1], item[0]))[:top_k]
    results = []
    for rank, (doc_id, score) in enumerate(ranking, 1):
        document = documents.iloc[document_to_row[doc_id]]
        results.append({"rank": rank, "document_id": doc_id, "score": float(score),
                        "title": str(document["title"]), "snippet": str(document["content"])[:240]})
    return {"model": "bm25", "results": results}


### Ejemplo de consulta
Los scores de TF-IDF y BM25 tienen escalas diferentes; se comparan rankings y métricas, no sus magnitudes.


In [67]:
query = "Does vitamin D reduce cancer risk?"
for search in [buscar_tfidf, buscar_bm25]:
    response = search(query, top_k=3)
    print("Modelo:", response["model"], "| Consulta:", query)
    display(pd.DataFrame(response["results"]))


Modelo: tfidf | Consulta: Does vitamin D reduce cancer risk?


,rank,document_id,score,title,snippet
0,1,MED-4573,0.518887,"Low Vitamin D Status: Definition, Prevalence, ...","Low Vitamin D Status: Definition, Prevalence, ..."
1,2,MED-716,0.496763,VITAMIN D: A D-LIGHTFUL SOLUTION FOR HEALTH,VITAMIN D: A D-LIGHTFUL SOLUTION FOR HEALTH Th...
2,3,MED-3990,0.486219,Vitamin D supplementation for prevention of mo...,Vitamin D supplementation for prevention of mo...


Modelo: bm25 | Consulta: Does vitamin D reduce cancer risk?


,rank,document_id,score,title,snippet
0,1,MED-4570,15.044376,Vitamin D supplement doses and serum 25-hydrox...,Vitamin D supplement doses and serum 25-hydrox...
1,2,MED-716,13.872971,VITAMIN D: A D-LIGHTFUL SOLUTION FOR HEALTH,VITAMIN D: A D-LIGHTFUL SOLUTION FOR HEALTH Th...
2,3,MED-3799,13.359881,Low-fat dietary pattern and risk of benign pro...,Low-fat dietary pattern and risk of benign pro...


## 5. Evaluación con test
Se usan **todas las consultas de `qrels/test.tsv`**, sin ajustar parámetros con test.
Para Precision y Recall, relevante significa `score > 0`:

- Precision@10: relevantes en las diez primeras posiciones / 10.
- Recall@10: relevantes en esas posiciones / total de relevantes conocidos.
- NDCG@10: DCG / DCG ideal, con $DCG=\sum_i(2^{rel_i}-1)/\log_2(i+1)$.

El ideal ordena todos los qrels de la consulta por grado. Documentos sin juicio y
posiciones faltantes aportan cero; los denominadores nulos se resuelven con cero.
Los promedios dan igual peso a cada consulta.


In [68]:
def evaluar_ranking(results, judgments, k=10):
    relevant = {doc_id for doc_id, grade in judgments.items() if grade > 0}
    ids = [row["document_id"] for row in results[:k]]
    hits = sum(doc_id in relevant for doc_id in ids)
    grades = [judgments.get(doc_id, 0) for doc_id in ids]
    dcg = sum((2 ** float(g) - 1) / np.log2(i + 1) for i, g in enumerate(grades, 1))
    ideal = sorted(judgments.values(), reverse=True)[:k]
    idcg = sum((2 ** float(g) - 1) / np.log2(i + 1) for i, g in enumerate(ideal, 1))
    return {"Precision@10": hits / k,
            "Recall@10": hits / len(relevant) if relevant else 0.,
            "NDCG@10": dcg / idcg if idcg else 0.}

queries_by_id = queries.set_index("_id")["text"].to_dict()
judgments_test = {str(qid): dict(zip(group["corpus-id"].astype(str), group["score"]))
                  for qid, group in qrels["test"].groupby("query-id", sort=True)}
used_queries = pd.DataFrame({"query_id": list(judgments_test)})
used_queries["Consulta"] = used_queries["query_id"].map(queries_by_id)
assert used_queries["Consulta"].notna().all()
print("Consultas utilizadas: todas las de qrels/test.tsv. Total:", len(used_queries))

rows = []
for query_id, judgments in judgments_test.items():
    query = queries_by_id[query_id]
    for model, search in [("tfidf", buscar_tfidf), ("bm25", buscar_bm25)]:
        results = search(query, top_k=10)["results"]
        rows.append({"query_id": query_id, "Modelo": model, **evaluar_ranking(results, judgments)})
per_query_metrics = pd.DataFrame(rows)
metrics = ["Precision@10", "Recall@10", "NDCG@10"]
summary_metrics = per_query_metrics.groupby("Modelo")[metrics].mean()
print("Promedios de test:")
display(summary_metrics)
for metric in metrics:
    difference = summary_metrics.loc["bm25", metric] - summary_metrics.loc["tfidf", metric]
    winner = "BM25" if difference > 0 else "TF-IDF" if difference < 0 else "Empate"
    print(f"{metric}: {winner}; diferencia BM25 menos TF-IDF = {difference:+.6f}")


Consultas utilizadas: todas las de qrels/test.tsv. Total: 323
Promedios de test:


,Precision@10,Recall@10,NDCG@10
Modelo,,,
bm25,0.239319,0.158096,0.330443
tfidf,0.228793,0.151456,0.312906


Precision@10: BM25; diferencia BM25 menos TF-IDF = +0.010526
Recall@10: BM25; diferencia BM25 menos TF-IDF = +0.006640
NDCG@10: BM25; diferencia BM25 menos TF-IDF = +0.017537


### Interpretación
BM25 supera a TF-IDF en los promedios de las tres métricas para esta configuración,
aunque no necesariamente en cada consulta. BM25 satura la frecuencia y ajusta por
longitud; TF-IDF usa frecuencia cruda y norma del vector. Recall indica que aún
quedan relevantes fuera del top-10. Los qrels pueden ser incompletos: ausencia de
juicio positivo no demuestra irrelevancia real.


## 6. Servicio web
FastAPI expone `GET /health` y `POST /search` con `query`, `model` (`tfidf` o `bm25`)
y `top_k`. Acepta consultas nuevas y devuelve los campos del enunciado.
`motor.py` contiene las mismas funciones de recuperación y `api.py` permite iniciar
el servicio fuera de Jupyter. Los índices se construyen una vez al arrancar.


In [69]:
from typing import Literal
from fastapi import FastAPI
from pydantic import BaseModel, Field, field_validator

app = FastAPI(title="NFCorpus - TF-IDF y BM25")

class SearchRequest(BaseModel):
    query: str
    model: Literal["bm25", "tfidf"]
    top_k: int = Field(default=10, ge=1, strict=True)

    @field_validator("query")
    @classmethod
    def validate_query(cls, value):
        if not value.strip():
            raise ValueError("query no puede estar vacía")
        return value.strip()

@app.get("/health")
def health():
    return {"status": "ok", "corpus": "nfcorpus", "documents": N}

@app.post("/search")
def search(request: SearchRequest):
    function = buscar_bm25 if request.model == "bm25" else buscar_tfidf
    return function(request.query, request.top_k)


### Comprobación de endpoints
Probamos las rutas y respuestas con el cliente de FastAPI, dentro del proceso.
Esta prueba no publica el servicio ni demuestra acceso desde internet.


In [70]:
from fastapi.testclient import TestClient

with TestClient(app) as client:
    health = client.get("/health")
    assert health.status_code == 200
    print("GET /health:", health.json())
    for model in ["tfidf", "bm25"]:
        payload = {"query": "Exercise and blood pressure", "model": model, "top_k": 3}
        response = client.post("/search", json=payload)
        assert response.status_code == 200
        body = response.json()
        results = body["results"]
        assert body["model"] == model and len(results) == 3
        assert len({r["document_id"] for r in results}) == 3
        assert all(r["document_id"] in document_ids for r in results)
        assert results == sorted(results, key=lambda r: (-r["score"], r["document_id"]))
        print("POST /search:", model)
        display(pd.DataFrame(results))


GET /health: {'status': 'ok', 'corpus': 'nfcorpus', 'documents': 3633}
POST /search: tfidf


,rank,document_id,score,title,snippet
0,1,MED-1492,0.530747,The effects of blood pressure reduction and of...,The effects of blood pressure reduction and of...
1,2,MED-1192,0.381215,Use of blood pressure lowering drugs in the pr...,Use of blood pressure lowering drugs in the pr...
2,3,MED-2042,0.373130,Position statement. Part one: Immune function ...,Position statement. Part one: Immune function ...


POST /search: bm25


,rank,document_id,score,title,snippet
0,1,MED-4504,12.494419,Dietary nitrate supplementation reduces the O2...,Dietary nitrate supplementation reduces the O2...
1,2,MED-5329,12.322196,Rapid reduction of serum cholesterol and blood...,Rapid reduction of serum cholesterol and blood...
2,3,MED-1520,11.736408,The effects of peppermint on exercise performance,The effects of peppermint on exercise performa...


## 7. Servicio publicado

**URL pública de entrega:** https://nfcorpus-api.vercel.app

- Disponibilidad: https://nfcorpus-api.vercel.app/health
- Documentación y pruebas: https://nfcorpus-api.vercel.app/docs
- Búsqueda: `POST https://nfcorpus-api.vercel.app/search`

En Postman, seleccionar POST y Body → raw → JSON:

```json
{"query": "Exercise and blood pressure", "model": "bm25", "top_k": 10}
```

Usar `Content-Type: application/json`. Repetir con `tfidf` y consultas nuevas.
Verificado el 18/11/2026: health y búsquedas con ambos modelos responden HTTP 200;
los resultados comprobados respetan top-k, orden e IDs sin duplicados.
Mantener el despliegue disponible durante la revisión del docente.

Código del servicio: https://github.com/Handresth404/nfcorpus-api

Para probarlo localmente

```bash
python -m uvicorn api:app --host 0.0.0.0 --port 8000
```
